In [ ]:
import random
import numpy as np
import torch

from datasets import load_dataset
from transformers import DistilBertTokenizerFast,DistilBertForSequenceClassification,Trainer,TrainingArguments

from sklearn.metrics import accuracy_score, f1_score

In [9]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)



Load IMDB Dataset

In [10]:
dataset = load_dataset("imdb")

dataset



DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

DistilBERT tokenizer

In [11]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )



In [12]:
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Rename label column to labels
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

# Remove unnecessary columns
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

# Set PyTorch format
tokenized_dataset.set_format("torch")

tokenized_dataset



Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 50000
    })
})

DistilBERT model

In [13]:
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased",num_labels=2)



config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Accuracy, F1-score

In [14]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "f1": f1
    }



In [15]:
training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=5e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=2,

    weight_decay=0.01,

    logging_dir="./logs",
    logging_steps=100,

    seed=42,

    load_best_model_at_end=True,

    report_to="none"
)



`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


train model

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],

    compute_metrics=compute_metrics
)


trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.257406,0.268599,0.885520,0.875727
2,0.134569,0.306508,0.912520,0.912383


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=3126, training_loss=0.22106464063213638, metrics={'train_runtime': 1743.3784, 'train_samples_per_second': 28.68, 'train_steps_per_second': 1.793, 'total_flos': 3311684966400000.0, 'train_loss': 0.22106464063213638, 'epoch': 2.0})

Evaluation

In [18]:
results = trainer.evaluate()

results



{'eval_loss': 0.2684539556503296,
 'eval_accuracy': 0.8856,
 'eval_f1': 0.8758249392150053,
 'eval_runtime': 213.5145,
 'eval_samples_per_second': 117.088,
 'eval_steps_per_second': 7.32,
 'epoch': 2.0}

save result

In [19]:
with open("training_results.txt", "w") as f:
    f.write("DistilBERT Fine-Tuning Results on IMDB Dataset\n")

    for key, value in results.items():
        f.write(f"{key}: {value}\n")



save model

In [21]:
save_path = "/content/fine_tuned_model/distilbert-imdb/"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)




Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/fine_tuned_model/distilbert-imdb/tokenizer_config.json',
 '/content/fine_tuned_model/distilbert-imdb/tokenizer.json')

Test Model on Custom Sentences

In [22]:
test_sentences = [
    "The movie was boring.",
    "I loved every part of it!"
]

model.eval()

for sentence in test_sentences:

    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )
    if torch.cuda.is_available():
        model.cuda()
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()

    if prediction == 1:
      sentiment = "Positive"
    else:
      sentiment = "Negative"

    print(f"Sentence: {sentence}")
    print(f"Predicted Sentiment: {sentiment}")




Sentence: The movie was boring.
Predicted Sentiment: Negative
Sentence: I loved every part of it!
Predicted Sentiment: Positive
